In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="04-youtube-video-search",
    notebook_name="03_ranking_and_serving.ipynb"
)

# YouTube Video Search: Ranking and Serving

## The Big Idea (For a 12-Year-Old)

Imagine you are a school librarian and 500 students ask you for books at the same time. You cannot carefully read every book for every student -- that would take forever. Instead, you:

1. **First pass (Candidate Generation)**: Quickly grab 100 books from the shelves that MIGHT be good. This takes 1 second.
2. **Second pass (Ranking)**: Look at those 100 books more carefully and pick the 10 best ones. This takes 5 seconds.
3. **Final touch (Re-ranking)**: Apply some common-sense rules -- remove books that are checked out, put new arrivals first, make sure you are not giving the same author five times. This takes 0.5 seconds.

This is EXACTLY how YouTube Video Search works at scale. The system searches **1 billion videos** and returns results in **under 200 milliseconds**. How? By using this multi-stage funnel.

---

## Staff-Level Technical Summary

We cover the complete post-retrieval pipeline:
- **Candidate generation**: Dual-path retrieval (ANN + Elasticsearch) producing ~1000 candidates
- **Learning to rank (LTR)**: Pointwise, pairwise, and listwise approaches with implementations
- **Re-ranking with business rules**: Freshness, diversity, content policy
- **Serving infrastructure**: Feature store, model serving, caching
- **Latency optimization**: Quantization, pruning, batching
- **A/B testing**: Online experimentation framework
- **Feature store architecture**: Online + offline feature serving

## 1. The Multi-Stage Funnel: From 1 Billion to 10

### Why Multi-Stage?

**Simple explanation**: You cannot run a super-fancy model on 1 billion videos -- that would take minutes. But you CAN run a simple model on 1 billion to get 1000, then a fancy model on 1000 to get 10.

**Staff-level insight**: This is the classic speed-accuracy tradeoff. Each stage is:
- **Wider but dumber** (earlier stages) or
- **Narrower but smarter** (later stages)

```
                    1,000,000,000 videos
                           |
               [Candidate Generation] ~10ms
                  Simple models, ANN
                           |
                     ~1,000 videos
                           |
                   [Ranking / Scoring] ~50ms
                  Complex ML model (LTR)
                           |
                      ~100 videos
                           |
                    [Re-Ranking] ~5ms
                  Business rules, diversity
                           |
                      10-20 videos
                     (shown to user)
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# === Visualize the Multi-Stage Funnel ===

fig, ax = plt.subplots(figsize=(12, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# Draw funnel shape
stages = [
    {'y': 8.5, 'width': 9, 'label': '1 Billion Videos', 'color': '#E3F2FD', 'items': '1,000,000,000'},
    {'y': 6.5, 'width': 6, 'label': 'Candidate Generation\n(ANN + Elasticsearch)', 'color': '#BBDEFB', 'items': '~1,000'},
    {'y': 4.5, 'width': 4, 'label': 'Ranking (LTR Model)\nPointwise/Pairwise/Listwise', 'color': '#90CAF9', 'items': '~100'},
    {'y': 2.5, 'width': 2.5, 'label': 'Re-Ranking\nBusiness Rules + Diversity', 'color': '#64B5F6', 'items': '~10-20'},
    {'y': 0.5, 'width': 1.5, 'label': 'Final Results', 'color': '#42A5F5', 'items': 'Displayed'},
]

latencies = ['', '~10ms', '~50ms', '~5ms', '']
model_complexity = ['', 'Simple (dot product)', 'Complex (neural LTR)', 'Rules engine', '']

for i, stage in enumerate(stages):
    x_center = 5
    w = stage['width']
    rect = mpatches.FancyBboxPatch((x_center - w/2, stage['y']), w, 1.2,
                                   boxstyle='round,pad=0.1',
                                   facecolor=stage['color'], edgecolor='#1565C0', linewidth=2)
    ax.add_patch(rect)
    ax.text(x_center, stage['y'] + 0.6, stage['label'],
            ha='center', va='center', fontsize=10, fontweight='bold')
    
    # Right side annotations
    if latencies[i]:
        ax.text(x_center + w/2 + 0.3, stage['y'] + 0.8, f'Latency: {latencies[i]}',
                fontsize=9, color='#E65100', fontstyle='italic')
        ax.text(x_center + w/2 + 0.3, stage['y'] + 0.3, f'Model: {model_complexity[i]}',
                fontsize=8, color='#555')
    
    # Left side item count
    ax.text(x_center - w/2 - 0.3, stage['y'] + 0.6, stage['items'],
            ha='right', fontsize=10, fontweight='bold', color='#1565C0')
    
    # Arrow to next stage
    if i < len(stages) - 1:
        ax.annotate('', xy=(x_center, stage['y']), xytext=(x_center, stage['y'] - 0.3),
                    arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))

ax.set_title('Multi-Stage Search Funnel: From 1 Billion to 10', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('Key insight: Each stage filters 10-100x, allowing increasingly expensive models.')
print('Total latency: 10 + 50 + 5 = ~65ms (well within the 200ms budget).')

## 2. Candidate Generation Pipeline

### The Two Librarians (Dual-Path Retrieval)

Remember from Notebook 01 -- we have two search paths:

1. **Visual Search Path (ANN)**: Encodes the text query with BERT, finds videos with similar embeddings using Approximate Nearest Neighbor search
2. **Text Search Path (Elasticsearch)**: Matches query words against video titles, descriptions, and tags using an inverted index

Each path returns ~500 candidates. Combined, we get ~1000 unique candidates (with some overlap).

### Why Both Paths?

| Scenario | Visual Path Wins | Text Path Wins |
|----------|------------------|----------------|
| Video titled "VID_001.mp4" showing puppies | Yes (understands visual content) | No (bad title) |
| Video titled "Best Python Tutorial" (talking head) | No (generic visual) | Yes (great title) |
| Video with both good title AND relevant content | Both contribute | Both contribute |

In [ ]:
# === Candidate Generation Simulation ===

np.random.seed(42)

class CandidateGenerator:
    """
    Simulates the dual-path candidate generation.
    
    12-year-old version:
    Two librarians working in parallel:
    - Librarian A watches all the videos and knows what they LOOK like
    - Librarian B reads all the titles and knows what they SAY
    Both grab their best candidates, then we merge the lists.
    """
    def __init__(self, n_videos=10000, embedding_dim=256):
        self.n_videos = n_videos
        self.embedding_dim = embedding_dim
        
        # Simulate pre-computed video embeddings (from offline indexing)
        self.video_embeddings = np.random.randn(n_videos, embedding_dim)
        self.video_embeddings /= np.linalg.norm(self.video_embeddings, axis=1, keepdims=True)
        
        # Simulate text index (inverted index)
        categories = ['music', 'sports', 'cooking', 'tech', 'comedy',
                      'education', 'travel', 'gaming', 'news', 'fitness']
        self.video_categories = [categories[i % len(categories)] for i in range(n_videos)]
    
    def visual_search(self, query_embedding, top_k=500):
        """ANN search: O(log N) with HNSW/FAISS."""
        similarities = self.video_embeddings @ query_embedding
        top_indices = np.argsort(-similarities)[:top_k]
        return [(idx, similarities[idx]) for idx in top_indices]
    
    def text_search(self, query_category, top_k=500):
        """Elasticsearch inverted index lookup."""
        matching = [(i, 1.0) for i, cat in enumerate(self.video_categories)
                    if cat == query_category]
        # Add noise scores
        matching = [(idx, score + np.random.uniform(0, 0.5)) for idx, score in matching]
        matching.sort(key=lambda x: -x[1])
        return matching[:top_k]
    
    def generate_candidates(self, query_embedding, query_category, top_k=1000):
        """Merge results from both paths."""
        visual_results = self.visual_search(query_embedding, top_k=500)
        text_results = self.text_search(query_category, top_k=500)
        
        # Merge with deduplication
        candidate_scores = {}
        for idx, score in visual_results:
            candidate_scores[idx] = {'visual': score, 'text': 0.0}
        for idx, score in text_results:
            if idx in candidate_scores:
                candidate_scores[idx]['text'] = score
            else:
                candidate_scores[idx] = {'visual': 0.0, 'text': score}
        
        # Sort by combined score
        combined = [(idx, 0.6 * s['visual'] + 0.4 * s['text'])
                    for idx, s in candidate_scores.items()]
        combined.sort(key=lambda x: -x[1])
        
        return combined[:top_k], len(visual_results), len(text_results)


# Demo
gen = CandidateGenerator(n_videos=10000)
query_emb = np.random.randn(256)
query_emb /= np.linalg.norm(query_emb)

candidates, n_visual, n_text = gen.generate_candidates(query_emb, 'music', top_k=1000)

print('=== Candidate Generation Results ===')
print(f'  Visual path returned: {n_visual} candidates')
print(f'  Text path returned:   {n_text} candidates')
print(f'  Merged (deduplicated): {len(candidates)} candidates')
print(f'\nTop 5 candidates:')
for rank, (idx, score) in enumerate(candidates[:5], 1):
    print(f'  #{rank}: Video {idx}, combined score = {score:.4f}')

print(f'\nThese ~1000 candidates now go to the RANKING stage.')
print(f'We traded coverage (billion -> thousand) for the ability')
print(f'to run a much more expensive ranking model.')

## 3. Learning to Rank (LTR)

### The Report Card Analogy

Think of ranking as giving each candidate video a "report card" score. There are three philosophies for how to grade:

1. **Pointwise**: Grade each video independently. "Is this video relevant? Give it a score from 0 to 1." Like grading each student's test without looking at other tests.

2. **Pairwise**: Compare videos in pairs. "Is Video A better than Video B?" Like a round-robin tournament.

3. **Listwise**: Grade the entire list at once. "What is the best ordering of all these videos?" Like ranking all students by their combined performance.

| Approach | Input | Output | Loss | Pros | Cons |
|----------|-------|--------|------|------|------|
| **Pointwise** | Single (query, video) | Relevance score | MSE or cross-entropy | Simple, fast | Ignores relative ordering |
| **Pairwise** | Pair of (query, videoA, videoB) | Which is better? | Hinge loss, RankNet | Learns relative order | O(n^2) pairs, slow |
| **Listwise** | List of (query, [videos]) | Best permutation | ListMLE, LambdaRank | Directly optimizes ranking | Complex, needs full lists |

In [ ]:
# === Learning to Rank: All Three Approaches ===

# First, let's create shared feature representation

class VideoFeatureExtractor:
    """
    Extract features for ranking.
    
    12-year-old version:
    For each (query, video) pair, we compute a bunch of numbers
    that describe how well they match. Things like:
    - How similar are their embeddings?
    - How popular is the video?
    - How fresh is the video?
    - Does the title match the query?
    """
    @staticmethod
    def extract(query_emb, video_emb, video_meta):
        features = [
            np.dot(query_emb, video_emb),            # Embedding similarity
            video_meta.get('view_count_log', 0),      # Popularity (log scale)
            video_meta.get('freshness', 0),            # Days since upload (normalized)
            video_meta.get('title_match_score', 0),    # BM25 text match
            video_meta.get('channel_authority', 0),    # Channel quality
            video_meta.get('video_length_bucket', 0),  # Duration category
            video_meta.get('engagement_rate', 0),      # Likes / views
            video_meta.get('ctr_historical', 0),       # Historical CTR
        ]
        return np.array(features)


# Generate synthetic ranking data
def generate_ranking_data(n_queries=200, n_candidates_per_query=50, n_features=8):
    """
    Generate synthetic (query, candidate_videos, relevance_labels) triples.
    """
    X_all = []
    y_all = []
    groups = []
    
    for q in range(n_queries):
        # Features for each candidate
        features = np.random.randn(n_candidates_per_query, n_features)
        # Make first feature (embedding similarity) more correlated with relevance
        relevance = np.clip(features[:, 0] * 0.5 + features[:, 1] * 0.2 + np.random.randn(n_candidates_per_query) * 0.3, 0, 5)
        relevance = np.round(relevance).astype(int)
        relevance = np.clip(relevance, 0, 5)
        
        X_all.append(features)
        y_all.append(relevance)
        groups.append(n_candidates_per_query)
    
    return np.vstack(X_all), np.concatenate(y_all), groups

X_train, y_train, groups = generate_ranking_data(n_queries=200)

print('=== Ranking Feature Data ===')
print(f'Total (query, video) pairs: {len(X_train)}')
print(f'Features per pair: {X_train.shape[1]}')
print(f'Relevance labels: {sorted(set(y_train))}')
print(f'\nFeature names:')
feature_names = ['emb_similarity', 'view_count_log', 'freshness', 'title_match',
                 'channel_authority', 'video_length', 'engagement_rate', 'ctr_historical']
for i, name in enumerate(feature_names):
    print(f'  [{i}] {name}: mean={X_train[:, i].mean():.3f}, std={X_train[:, i].std():.3f}')

In [ ]:
# === Approach 1: Pointwise Ranking ===

class PointwiseRanker(nn.Module):
    """
    Treats ranking as a regression/classification problem.
    
    12-year-old version:
    Look at each video ONE AT A TIME and ask: 'How relevant is this?'
    Give it a score from 0 to 5. Like grading homework papers
    independently without comparing them to each other.
    
    Staff-level detail:
    - Maps features to a relevance score using a neural network
    - Loss: MSE (regression) or cross-entropy (classification)
    - Fast: O(n) per query, where n = number of candidates
    - Weakness: does not directly optimize ranking order
    """
    def __init__(self, n_features, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 1)  # Single relevance score
        )
    
    def forward(self, x):
        return self.net(x).squeeze(-1)


# Train pointwise ranker
pointwise_model = PointwiseRanker(n_features=8)
optimizer = torch.optim.Adam(pointwise_model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

X_tensor = torch.FloatTensor(X_train)
y_tensor = torch.FloatTensor(y_train)

pointwise_losses = []
for epoch in range(100):
    optimizer.zero_grad()
    predictions = pointwise_model(X_tensor)
    loss = criterion(predictions, y_tensor)
    loss.backward()
    optimizer.step()
    pointwise_losses.append(loss.item())
    if (epoch + 1) % 25 == 0:
        print(f'  Epoch {epoch+1}: Loss = {loss.item():.4f}')

print(f'\nPointwise approach:')
print(f'  Strength: Simple, fast O(n), easy to train')
print(f'  Weakness: Optimizes absolute scores, NOT relative ranking')
print(f'  Use when: You have large datasets and need fast iteration')

In [ ]:
# === Approach 2: Pairwise Ranking (RankNet) ===

class PairwiseRanker(nn.Module):
    """
    RankNet-style pairwise learning to rank.
    
    12-year-old version:
    Instead of grading each video alone, we compare them in PAIRS:
    'Is Video A better than Video B for this query?'
    Like a tournament bracket -- you learn who beats whom.
    
    Staff-level detail:
    - Compute scores for both items: s_i = f(x_i), s_j = f(x_j)
    - Probability that item i is ranked higher: P(i > j) = sigmoid(s_i - s_j)
    - Loss: cross-entropy between predicted P(i > j) and true label
    - O(n^2) pairs per query, but can sample for efficiency
    - Better than pointwise at capturing relative ordering
    """
    def __init__(self, n_features, hidden_dim=64):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.Linear(n_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, x):
        return self.scorer(x).squeeze(-1)
    
    def pairwise_loss(self, x_i, x_j, y_ij):
        """
        x_i: features of the higher-relevance item
        x_j: features of the lower-relevance item
        y_ij: 1 if item i should rank higher than j
        """
        s_i = self.forward(x_i)
        s_j = self.forward(x_j)
        diff = s_i - s_j
        loss = F.binary_cross_entropy_with_logits(diff, y_ij)
        return loss


# Generate pairwise training data from pointwise labels
def create_pairs(X, y, groups, max_pairs_per_query=100):
    """Create (better_item, worse_item) pairs from relevance labels."""
    X_better, X_worse, labels = [], [], []
    offset = 0
    for g in groups:
        indices = list(range(offset, offset + g))
        pairs_created = 0
        for i in indices:
            for j in indices:
                if y[i] > y[j] and pairs_created < max_pairs_per_query:
                    X_better.append(X[i])
                    X_worse.append(X[j])
                    labels.append(1.0)
                    pairs_created += 1
        offset += g
    return (torch.FloatTensor(np.array(X_better)),
            torch.FloatTensor(np.array(X_worse)),
            torch.FloatTensor(np.array(labels)))

X_better, X_worse, pair_labels = create_pairs(X_train, y_train, groups)

pairwise_model = PairwiseRanker(n_features=8)
optimizer = torch.optim.Adam(pairwise_model.parameters(), lr=1e-3)

pairwise_losses = []
batch_size = 512
for epoch in range(100):
    # Mini-batch training
    perm = torch.randperm(len(pair_labels))[:batch_size]
    optimizer.zero_grad()
    loss = pairwise_model.pairwise_loss(X_better[perm], X_worse[perm], pair_labels[perm])
    loss.backward()
    optimizer.step()
    pairwise_losses.append(loss.item())
    if (epoch + 1) % 25 == 0:
        print(f'  Epoch {epoch+1}: Pairwise Loss = {loss.item():.4f}')

print(f'\nPairwise (RankNet) approach:')
print(f'  Total training pairs: {len(pair_labels)}')
print(f'  Strength: Directly learns relative ordering')
print(f'  Weakness: O(n^2) pairs per query -- expensive')
print(f'  Use when: Quality matters more than training speed')

In [ ]:
# === Approach 3: Listwise Ranking (ListMLE) ===

class ListwiseRanker(nn.Module):
    """
    Listwise learning to rank using ListMLE loss.
    
    12-year-old version:
    Instead of grading one video or comparing two, we look at
    the ENTIRE list of candidates and ask: 'What is the best
    ordering of ALL of these?'
    
    Staff-level detail:
    ListMLE loss:
    - Define the ideal permutation based on ground truth relevance
    - Compute the probability of the model producing this permutation
    - Maximize this probability (minimize negative log-likelihood)
    
    P(permutation) = PRODUCT_i  exp(s_pi(i)) / SUM_j>=i exp(s_pi(j))
    
    This directly optimizes the ranking of the full list.
    """
    def __init__(self, n_features, hidden_dim=64):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.Linear(n_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, x):
        return self.scorer(x).squeeze(-1)
    
    def listmle_loss(self, scores, relevance):
        """
        ListMLE loss for a single query's candidate list.
        
        scores: (n_candidates,) predicted scores
        relevance: (n_candidates,) ground truth relevance
        """
        # Get ideal permutation (sort by relevance, descending)
        ideal_order = torch.argsort(relevance, descending=True)
        sorted_scores = scores[ideal_order]
        
        # ListMLE: log probability of ideal permutation
        n = len(sorted_scores)
        loss = 0.0
        for i in range(n - 1):
            # log-sum-exp of remaining items
            log_sum = torch.logsumexp(sorted_scores[i:], dim=0)
            loss += log_sum - sorted_scores[i]
        
        return loss / n


# Train listwise ranker
listwise_model = ListwiseRanker(n_features=8)
optimizer = torch.optim.Adam(listwise_model.parameters(), lr=1e-3)

listwise_losses = []
n_queries = len(groups)

for epoch in range(100):
    epoch_loss = 0
    offset = 0
    for g in groups[:50]:  # Use subset for speed
        x_query = torch.FloatTensor(X_train[offset:offset + g])
        y_query = torch.FloatTensor(y_train[offset:offset + g])
        
        optimizer.zero_grad()
        scores = listwise_model(x_query)
        loss = listwise_model.listmle_loss(scores, y_query)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        offset += g
    
    avg_loss = epoch_loss / 50
    listwise_losses.append(avg_loss)
    if (epoch + 1) % 25 == 0:
        print(f'  Epoch {epoch+1}: Listwise Loss = {avg_loss:.4f}')

print(f'\nListwise (ListMLE) approach:')
print(f'  Strength: Directly optimizes the full ranking order')
print(f'  Weakness: More complex, needs full query groups')
print(f'  Use when: You need the best possible ranking quality')

In [ ]:
# === Compare All Three Approaches ===

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, losses, name, color in [
    (axes[0], pointwise_losses, 'Pointwise (MSE)', '#2196F3'),
    (axes[1], pairwise_losses, 'Pairwise (RankNet)', '#4CAF50'),
    (axes[2], listwise_losses, 'Listwise (ListMLE)', '#FF9800'),
]:
    ax.plot(losses, color=color, linewidth=2)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Loss', fontsize=11)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('Learning to Rank: Three Approaches Compared', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Summary table
print('=== LTR Approach Comparison ===')
print(f'{"Approach":<15} {"Training Cost":<18} {"Quality":<12} {"Best For":<30}')
print('-' * 75)
print(f'{"Pointwise":<15} {"O(n)":<18} {"Good":<12} {"Large-scale, fast iteration":<30}')
print(f'{"Pairwise":<15} {"O(n^2)":<18} {"Better":<12} {"Medium-scale, quality matters":<30}')
print(f'{"Listwise":<15} {"O(n log n)":<18} {"Best":<12} {"Small candidate sets, max quality":<30}')
print(f'\nRecommendation: Start with Pointwise (simple baseline),')
print(f'upgrade to Pairwise/Listwise when diminishing returns from features.')

## 4. Re-Ranking with Business Rules

### The Final Polish

**Simple explanation**: The ranking model gives each video a score based purely on relevance. But the business has other needs too:
- Do not show the same creator five times (diversity)
- Boost fresh content for news queries (freshness)
- Remove policy-violating videos (safety)
- Handle sponsored content placement (monetization)

The re-ranking layer applies these rules AFTER the ML model has done its work.

In [ ]:
# === Re-Ranking Engine ===

class ReRankingEngine:
    """
    Post-ML re-ranking with business rules.
    
    12-year-old version:
    The ML model picks the most relevant videos, but a human
    manager then adjusts the list:
    - 'We cannot show 5 videos from the same channel!'
    - 'This video about today's news should be boosted!'
    - 'This video was flagged -- remove it!'
    
    Staff-level detail:
    Each rule is a separate function with a configurable weight.
    Rules are applied sequentially (pipeline pattern).
    Weights are tuned via A/B testing, not training.
    """
    def __init__(self):
        self.rules = [
            ('policy_filter', self.filter_policy_violations),
            ('freshness_boost', self.apply_freshness_boost),
            ('diversity', self.enforce_diversity),
            ('dedup', self.remove_near_duplicates),
        ]
    
    def filter_policy_violations(self, results):
        """Remove flagged or policy-violating content."""
        return [r for r in results if not r.get('flagged', False)]
    
    def apply_freshness_boost(self, results, freshness_weight=0.1):
        """Boost recently uploaded videos."""
        for r in results:
            days_old = r.get('days_since_upload', 30)
            freshness_score = max(0, 1 - days_old / 365)
            r['score'] += freshness_weight * freshness_score
        return sorted(results, key=lambda x: -x['score'])
    
    def enforce_diversity(self, results, max_per_channel=2):
        """Limit videos from the same channel."""
        channel_counts = {}
        diverse_results = []
        for r in results:
            channel = r.get('channel', 'unknown')
            channel_counts[channel] = channel_counts.get(channel, 0) + 1
            if channel_counts[channel] <= max_per_channel:
                diverse_results.append(r)
        return diverse_results
    
    def remove_near_duplicates(self, results, sim_threshold=0.95):
        """Remove videos that are near-duplicates of higher-ranked ones."""
        kept = []
        for r in results:
            is_duplicate = False
            for k in kept:
                if r.get('content_hash', '') == k.get('content_hash', '_'):
                    is_duplicate = True
                    break
            if not is_duplicate:
                kept.append(r)
        return kept
    
    def rerank(self, results, top_k=10):
        """Apply all rules in sequence."""
        for rule_name, rule_fn in self.rules:
            before = len(results)
            results = rule_fn(results)
            after = len(results)
            if before != after:
                print(f'  Rule [{rule_name}]: {before} -> {after} results')
        return results[:top_k]


# Demo
np.random.seed(42)
channels = ['MusicHub', 'CookShow', 'TechDaily', 'MusicHub', 'MusicHub',
            'GamerX', 'TechDaily', 'FitnessGo', 'MusicHub', 'CookShow',
            'TechDaily', 'GamerX', 'FitnessGo', 'MusicHub', 'CookShow']

mock_results = [
    {'video_id': f'v_{i}', 'score': 1.0 - i * 0.05, 'channel': channels[i],
     'days_since_upload': np.random.randint(1, 200),
     'flagged': i == 3,  # One video is flagged
     'content_hash': f'hash_{i}'}
    for i in range(15)
]

engine = ReRankingEngine()
print('=== Before Re-Ranking ===')
for r in mock_results[:8]:
    flag = ' [FLAGGED]' if r['flagged'] else ''
    print(f"  {r['video_id']} | {r['channel']:<10} | score={r['score']:.2f} | {r['days_since_upload']}d old{flag}")

print('\n=== Applying Re-Ranking Rules ===')
final_results = engine.rerank(mock_results, top_k=8)

print('\n=== After Re-Ranking ===')
for i, r in enumerate(final_results, 1):
    print(f"  #{i}: {r['video_id']} | {r['channel']:<10} | score={r['score']:.3f} | {r['days_since_upload']}d old")

print('\nNotice: Flagged video removed, max 2 per channel enforced, freshness boosted.')

## 5. Serving Infrastructure Design

### The Restaurant Kitchen Analogy

A high-traffic restaurant needs:
- **Prep station** (Feature Store): Pre-chop vegetables so cooks do not waste time during rush hour
- **Multiple cooks** (Model Servers): Handle many orders in parallel
- **Pass window** (Load Balancer): Route orders to available cooks
- **Pre-made sauces** (Caching): Keep popular items ready to go

YouTube's search infrastructure works the same way.

In [ ]:
# === Serving Architecture Diagram ===

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')

def draw_box(ax, x, y, w, h, text, color='#E8F4FD', edge='#2196F3', fontsize=9):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor=edge, linewidth=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=fontsize, fontweight='bold')

def draw_arrow(ax, x1, y1, x2, y2, color='#333'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))

# User request
draw_box(ax, 0.5, 8.5, 2.5, 1, 'User Query\n"funny cats"', '#FFF3E0', '#FF9800', 10)

# API Gateway / Load Balancer
draw_box(ax, 4, 8.5, 3, 1, 'API Gateway\nLoad Balancer', '#E8EAF6', '#5C6BC0', 10)
draw_arrow(ax, 3, 9, 4, 9)

# Query Understanding
draw_box(ax, 8, 8.5, 3, 1, 'Query Understanding\nSpelling, Intent, Entities', '#F3E5F5', '#7B1FA2', 9)
draw_arrow(ax, 7, 9, 8, 9)

# Candidate Generation (two paths)
draw_box(ax, 0.5, 6, 3.5, 1.2, 'Visual Search (ANN)\nFAISS / ScaNN / HNSW', '#E3F2FD', '#1565C0', 9)
draw_box(ax, 5, 6, 3.5, 1.2, 'Text Search\nElasticsearch', '#E8F5E9', '#2E7D32', 9)
draw_arrow(ax, 9.5, 8.5, 2.5, 7.2)
draw_arrow(ax, 9.5, 8.5, 7, 7.2)

# Feature Store
draw_box(ax, 10, 6, 3, 1.2, 'Feature Store\n(Online + Offline)', '#FFF9C4', '#F9A825', 9)

# Ranking
draw_box(ax, 3, 3.8, 4, 1.2, 'Ranking Model (LTR)\nNeural Network Scorer', '#FFCCBC', '#BF360C', 10)
draw_arrow(ax, 2.5, 6, 5, 5)
draw_arrow(ax, 6.5, 6, 5, 5)
draw_arrow(ax, 11.5, 6, 7, 4.8, '#F9A825')

# Re-ranking
draw_box(ax, 8.5, 3.8, 3.5, 1.2, 'Re-Ranking Service\nDiversity, Freshness, Policy', '#E0F2F1', '#00695C', 9)
draw_arrow(ax, 7, 4.4, 8.5, 4.4)

# Model Registry
draw_box(ax, 13.5, 3.8, 2, 1.2, 'Model\nRegistry', '#FCE4EC', '#C62828', 9)
draw_arrow(ax, 13.5, 4.4, 12, 4.4, '#C62828')

# Result Cache
draw_box(ax, 3, 1.5, 3, 1, 'Result Cache\n(Redis / Memcached)', '#E8EAF6', '#5C6BC0', 9)

# Final Response
draw_box(ax, 7.5, 1.5, 3, 1, 'Final Results\nReturned to User', '#FFF3E0', '#FF9800', 10)
draw_arrow(ax, 10, 3.8, 9, 2.5)
draw_arrow(ax, 6, 1.5, 4.5, 2)

# Latency annotations
ax.text(2.5, 5.5, '~10ms', fontsize=9, color='#E65100', fontstyle='italic', ha='center')
ax.text(5, 3.3, '~50ms', fontsize=9, color='#E65100', fontstyle='italic', ha='center')
ax.text(10, 3.3, '~5ms', fontsize=9, color='#E65100', fontstyle='italic', ha='center')

ax.set_title('YouTube Video Search: Full Serving Architecture', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('Total end-to-end latency budget: ~65-80ms (well within 200ms target)')
print('Key components: API Gateway -> Query Understanding -> Dual-path retrieval')
print('  -> Feature Store -> LTR Ranking -> Re-ranking -> Cache -> Response')

## 6. Feature Store Architecture

### The Pantry Analogy

**Simple explanation**: A feature store is like a well-organized pantry in a restaurant. Some ingredients are prepared fresh (online features like "user's last 5 searches"), while others are batch-prepared (offline features like "video's total view count"). The feature store makes sure every ingredient is ready when the chef (model) needs it.

### Two Types of Features

| Type | Computation | Latency | Example |
|------|-------------|---------|----------|
| **Offline features** | Batch computed (hourly/daily) | Pre-computed, ~1ms lookup | Video view count, channel subscriber count |
| **Online features** | Computed in real-time | ~5-10ms | Query-video embedding similarity, user's recent searches |

In [ ]:
# === Feature Store Simulation ===

import time
from collections import OrderedDict

class FeatureStore:
    """
    Simulates an online/offline feature store.
    
    12-year-old version:
    Think of it as two shelves in a pantry:
    - Bottom shelf (offline): Big jars prepared yesterday.
      Pre-computed stats like 'this video has 1M views'.
    - Top shelf (online): Fresh ingredients prepared right now.
      Real-time stuff like 'how similar is this query to this video?'
    
    Staff-level detail:
    - Offline store: Redis/DynamoDB, populated by daily Spark/Flink jobs
    - Online store: Computed at request time, cached in local memory
    - Feature registry: Schema + lineage tracking for all features
    - Point-in-time correctness: Prevents data leakage in training
    """
    def __init__(self):
        # Offline features (pre-computed daily)
        self.offline_store = {}
        # Online feature cache (LRU)
        self.online_cache = OrderedDict()
        self.cache_max_size = 10000
        
        # Feature registry
        self.feature_registry = {
            'video_view_count_log': {'type': 'offline', 'update_freq': 'daily'},
            'video_like_ratio': {'type': 'offline', 'update_freq': 'daily'},
            'channel_subscriber_count_log': {'type': 'offline', 'update_freq': 'daily'},
            'video_age_days': {'type': 'offline', 'update_freq': 'daily'},
            'video_duration_seconds': {'type': 'offline', 'update_freq': 'once'},
            'query_video_emb_similarity': {'type': 'online', 'update_freq': 'per_request'},
            'query_video_title_bm25': {'type': 'online', 'update_freq': 'per_request'},
            'user_recent_search_similarity': {'type': 'online', 'update_freq': 'per_request'},
        }
    
    def populate_offline_features(self, video_ids):
        """Simulate daily batch job populating offline features."""
        for vid in video_ids:
            self.offline_store[vid] = {
                'video_view_count_log': np.log1p(np.random.randint(100, 10000000)),
                'video_like_ratio': np.random.uniform(0.7, 0.99),
                'channel_subscriber_count_log': np.log1p(np.random.randint(100, 5000000)),
                'video_age_days': np.random.randint(1, 365),
                'video_duration_seconds': np.random.randint(30, 3600),
            }
    
    def compute_online_features(self, query_emb, video_emb, query_text, video_title):
        """Compute features that require real-time data."""
        return {
            'query_video_emb_similarity': float(np.dot(query_emb, video_emb)),
            'query_video_title_bm25': float(len(set(query_text.lower().split()) & 
                                               set(video_title.lower().split()))),
            'user_recent_search_similarity': np.random.uniform(0, 1),  # Simplified
        }
    
    def get_features(self, video_id, query_emb=None, video_emb=None,
                     query_text='', video_title=''):
        """Get all features for a (query, video) pair."""
        features = {}
        
        # Offline features: fast lookup
        if video_id in self.offline_store:
            features.update(self.offline_store[video_id])
        
        # Online features: compute in real-time
        if query_emb is not None and video_emb is not None:
            online = self.compute_online_features(query_emb, video_emb,
                                                  query_text, video_title)
            features.update(online)
        
        return features


# Demo
store = FeatureStore()
video_ids = [f'v_{i}' for i in range(100)]
store.populate_offline_features(video_ids)

# Simulate a query-time feature fetch
query_emb = np.random.randn(256)
query_emb /= np.linalg.norm(query_emb)
video_emb = np.random.randn(256)
video_emb /= np.linalg.norm(video_emb)

features = store.get_features('v_42', query_emb, video_emb, 'funny cats', 'Cat Comedy Compilation')

print('=== Feature Store: Features for (query="funny cats", video="v_42") ===')
print(f'\n{"Feature":<35} {"Value":>10} {"Type":<12}')
print('-' * 60)
for feat_name, value in features.items():
    feat_type = store.feature_registry.get(feat_name, {}).get('type', 'unknown')
    print(f'  {feat_name:<33} {value:>10.4f} {feat_type:<12}')

print(f'\nOffline features: pre-computed daily, stored in Redis/DynamoDB')
print(f'Online features: computed per request in ~5ms')
print(f'Total feature fetch latency: ~5-10ms')

## 7. Latency Optimization

### The Race Car Analogy

Getting results in under 200ms is like a race. Every millisecond counts. Here are the tricks:

| Technique | What It Does | Savings |
|-----------|-------------|----------|
| **Model quantization** | Convert 32-bit floats to 8-bit integers | 4x smaller, 2-3x faster |
| **Model pruning** | Remove unimportant neurons | 30-50% fewer computations |
| **Embedding caching** | Cache popular query embeddings | Skip encoder entirely for hot queries |
| **Async I/O** | Run visual + text search in parallel | 2x faster than sequential |
| **Batch inference** | Process multiple candidates at once | Better GPU utilization |
| **Pre-filtering** | Remove unavailable videos before ranking | Fewer items to rank |

In [ ]:
# === Latency Optimization: Quantization Demo ===

class FullPrecisionScorer(nn.Module):
    """Full precision (FP32) ranking model."""
    def __init__(self, n_features=8, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1)
        )
    def forward(self, x):
        return self.net(x)

# Create models
fp32_model = FullPrecisionScorer()

# Quantize to INT8
fp32_model.eval()
int8_model = torch.quantization.quantize_dynamic(
    fp32_model, {nn.Linear}, dtype=torch.qint8
)

# Compare sizes
def model_size_bytes(model):
    import io
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    return buffer.tell()

fp32_size = model_size_bytes(fp32_model)
int8_size = model_size_bytes(int8_model)

# Compare inference speed
test_input = torch.randn(1000, 8)  # 1000 candidates

# FP32 timing
start = time.time()
for _ in range(100):
    with torch.no_grad():
        _ = fp32_model(test_input)
fp32_time = (time.time() - start) / 100

# INT8 timing
start = time.time()
for _ in range(100):
    with torch.no_grad():
        _ = int8_model(test_input)
int8_time = (time.time() - start) / 100

print('=== Model Quantization Comparison ===')
print(f'\n{"":<20} {"FP32":>12} {"INT8":>12} {"Improvement":>14}')
print('-' * 60)
print(f'{"Model size":<20} {fp32_size/1024:.1f} KB{"":>4} {int8_size/1024:.1f} KB{"":>4} {fp32_size/int8_size:.1f}x smaller')
print(f'{"Inference (1K items)":<20} {fp32_time*1000:.2f} ms{"":>3} {int8_time*1000:.2f} ms{"":>3} {fp32_time/int8_time:.1f}x faster')
print(f'\nQuantization reduces model size and inference time with minimal quality loss.')
print(f'Combined with async I/O and caching, we can serve results in ~65ms.')

In [ ]:
# === Latency Budget Breakdown ===

components = [
    ('Network (API Gateway)', 5),
    ('Query Understanding', 5),
    ('ANN Search (FAISS)', 10),
    ('Text Search (ES)', 12),
    ('Feature Fetch', 8),
    ('Ranking Model', 25),
    ('Re-Ranking', 3),
    ('Serialization', 2),
]

names = [c[0] for c in components]
latencies = [c[1] for c in components]
total = sum(latencies)

# Note: ANN and ES run in parallel, so actual total is less
parallel_savings = min(latencies[2], latencies[3])  # Parallel execution saves time
actual_total = total - parallel_savings

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = plt.cm.Set3(np.linspace(0, 1, len(names)))
bars = ax1.barh(range(len(names)), latencies, color=colors, edgecolor='black', linewidth=0.5)
ax1.set_yticks(range(len(names)))
ax1.set_yticklabels(names, fontsize=10)
ax1.set_xlabel('Latency (ms)', fontsize=11)
ax1.set_title('Latency Budget Per Component', fontsize=13, fontweight='bold')
ax1.invert_yaxis()
for bar, lat in zip(bars, latencies):
    ax1.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
             f'{lat}ms', va='center', fontsize=10)
ax1.axvline(x=actual_total, color='red', linestyle='--', linewidth=2,
            label=f'Total: ~{actual_total}ms (parallel ANN+ES)')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, axis='x')

# Pie chart of time allocation
ax2.pie(latencies, labels=names, colors=colors, autopct='%1.0f%%',
        textprops={'fontsize': 9}, startangle=90)
ax2.set_title(f'Latency Distribution\nTotal: ~{actual_total}ms (budget: 200ms)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Sequential total: {total}ms')
print(f'With parallel ANN + ES: ~{actual_total}ms')
print(f'Budget remaining: {200 - actual_total}ms headroom')
print(f'\nBiggest bottleneck: Ranking model ({latencies[5]}ms) -- optimize with quantization')

## 8. A/B Testing Strategies

### The Science Experiment Analogy

**Simple explanation**: Before replacing the old search system with the new one, we run an experiment. We show the old system to half the users (Control group) and the new system to the other half (Treatment group). After a week, we compare their behavior.

**Staff-level detail**: A/B testing for search systems requires careful design:

| Decision | Details |
|----------|----------|
| **Randomization unit** | User-level (not query-level) to avoid inconsistent experience |
| **Sample size** | Need statistical power calculation (typically 5-10% per variant) |
| **Duration** | 1-2 weeks minimum (captures day-of-week effects) |
| **Primary metric** | Total watch time (not CTR -- avoids clickbait) |
| **Guardrail metrics** | User satisfaction surveys, complaint rate, long sessions |
| **Novelty effects** | Users may initially engage more just because it is different |

In [ ]:
# === A/B Testing Framework ===

from scipy import stats

class ABTestAnalyzer:
    """
    Analyze A/B test results for search system changes.
    
    12-year-old version:
    You give half the school the old search engine and half
    the new one. After a week, you check:
    - Did the new group watch more videos? (engagement)
    - Did they click more results? (relevance)
    - Did they complain less? (satisfaction)
    If the new group did significantly better, ship the new version!
    """
    def __init__(self, confidence_level=0.95):
        self.confidence_level = confidence_level
    
    def run_test(self, control_values, treatment_values, metric_name):
        """Run a two-sample t-test."""
        control_mean = np.mean(control_values)
        treatment_mean = np.mean(treatment_values)
        lift = (treatment_mean - control_mean) / control_mean * 100
        
        t_stat, p_value = stats.ttest_ind(control_values, treatment_values)
        significant = p_value < (1 - self.confidence_level)
        
        return {
            'metric': metric_name,
            'control_mean': control_mean,
            'treatment_mean': treatment_mean,
            'lift_pct': lift,
            'p_value': p_value,
            'significant': significant,
        }


# Simulate A/B test data
np.random.seed(42)
n_users = 10000

# Control: old ranking model
control_watch_time = np.random.lognormal(4.0, 1.0, n_users)  # seconds per session
control_ctr = np.random.beta(3, 7, n_users)  # click-through rate
control_completion = np.random.beta(4, 6, n_users)  # video completion rate

# Treatment: new ranking model (slightly better)
treatment_watch_time = np.random.lognormal(4.05, 1.0, n_users)  # +5% effect
treatment_ctr = np.random.beta(3.2, 7, n_users)  # +3% effect
treatment_completion = np.random.beta(4.1, 6, n_users)  # +2% effect

analyzer = ABTestAnalyzer(confidence_level=0.95)

results = [
    analyzer.run_test(control_watch_time, treatment_watch_time, 'Total Watch Time (sec)'),
    analyzer.run_test(control_ctr, treatment_ctr, 'Click-Through Rate'),
    analyzer.run_test(control_completion, treatment_completion, 'Video Completion Rate'),
]

print('=== A/B Test Results: Old vs New Ranking Model ===')
print(f'Users per group: {n_users:,}')
print(f'Confidence level: 95%')
print()
print(f'{"Metric":<28} {"Control":>10} {"Treatment":>10} {"Lift":>8} {"p-value":>10} {"Sig?":>6}')
print('-' * 78)
for r in results:
    sig = 'YES' if r['significant'] else 'no'
    print(f"{r['metric']:<28} {r['control_mean']:>10.4f} {r['treatment_mean']:>10.4f} "
          f"{r['lift_pct']:>+7.2f}% {r['p_value']:>10.4f} {sig:>6}")

print(f'\nDecision: ', end='')
if all(r['significant'] and r['lift_pct'] > 0 for r in results):
    print('SHIP IT -- all metrics significantly improved.')
else:
    print('Review needed -- not all metrics are significant.')

In [ ]:
# === Visualize A/B Test Results ===

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

test_pairs = [
    (control_watch_time, treatment_watch_time, 'Watch Time (sec)', '#2196F3'),
    (control_ctr, treatment_ctr, 'Click-Through Rate', '#4CAF50'),
    (control_completion, treatment_completion, 'Completion Rate', '#FF9800'),
]

for ax, (ctrl, treat, name, color) in zip(axes, test_pairs):
    ax.hist(ctrl, bins=50, alpha=0.5, label='Control', color='#90CAF9', edgecolor='black', linewidth=0.3)
    ax.hist(treat, bins=50, alpha=0.5, label='Treatment', color=color, edgecolor='black', linewidth=0.3)
    ax.axvline(np.mean(ctrl), color='#1565C0', linestyle='--', linewidth=2, label=f'Control mean: {np.mean(ctrl):.3f}')
    ax.axvline(np.mean(treat), color='#BF360C', linestyle='--', linewidth=2, label=f'Treatment mean: {np.mean(treat):.3f}')
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('A/B Test: Distribution Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 9. Interview Cheat Sheet: Ranking and Serving

### Key Phrases to Use

| Topic | Key Phrase |
|-------|-----------|
| Multi-stage | "We use a funnel: candidate generation (cheap, broad) then ranking (expensive, precise) then re-ranking (business rules)" |
| LTR choice | "I would start with pointwise for fast iteration, then upgrade to pairwise/listwise for quality" |
| Feature store | "Offline features are batch-computed daily; online features are computed per-request in the hot path" |
| Latency | "Key techniques: model quantization, async parallel retrieval, result caching, pre-filtering" |
| A/B testing | "Randomize at user level, run for 1-2 weeks, watch guardrail metrics alongside the primary metric" |
| Re-ranking | "Business rules are applied post-ML: diversity, freshness, content policy, and sponsored content" |
| Scaling | "ANN search is sublinear; ranking model runs on GPU with batched inference" |

### Common Follow-Up Questions

1. **"How would you handle query intent classification?"** -- Use a separate classifier (news vs. entertainment vs. educational) to adjust freshness weights and source preferences.

2. **"What if the ranking model is too slow?"** -- Quantize INT8, prune 30% of weights, distill to a smaller student model, cache popular query embeddings.

3. **"How do you handle position bias in training data?"** -- Users click position 1 more than position 5 regardless of relevance. Use inverse propensity weighting or position-aware training.

4. **"How would you add personalization?"** -- Add user features (watch history embedding, demographic bucket) to the ranking model's feature vector.

5. **"What metrics would you watch for a regression?"** -- Guardrail metrics: search abandonment rate, time to first click, rage clicks (rapid re-searches).

## Key Takeaways

1. **Multi-stage funnel is essential**: 1B -> 1K (candidate gen) -> 100 (ranking) -> 10 (re-ranking). Each stage trades coverage for model complexity.

2. **Three LTR approaches**: Pointwise (simple, fast), Pairwise (learns relative order), Listwise (directly optimizes ranking). Start simple, upgrade as needed.

3. **Re-ranking is not optional**: Business rules (diversity, freshness, policy) are applied AFTER ML ranking. These rules are A/B tested, not trained.

4. **Feature store architecture**: Offline features (batch, daily) + Online features (real-time, per-request). Both feed into the ranking model.

5. **Latency optimization**: Quantization (INT8), parallel retrieval (ANN + ES async), caching (popular queries), batched inference (GPU efficiency).

6. **A/B testing**: Randomize at user level, run 1-2 weeks, primary metric = total watch time, guardrails = satisfaction + abandonment rate.

7. **Total latency**: ~65ms end-to-end (well within 200ms budget), with the ranking model being the biggest bottleneck (~25ms).

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)